In [1]:
import pygame
import random
import time
import urllib.request
import io

pygame.init()
pygame.font.init()

# Tela
LARGURA, ALTURA = 800, 600
tela = pygame.display.set_mode((LARGURA, ALTURA))
pygame.display.set_caption("Space Math Blaster 2D")

# Cores Modernas (Saindo do P&B)
PRETO = (10, 10, 25)
BRANCO = (255, 255, 255)
AZUL_PROPULSAO = (0, 230, 255)
LARANJA_FOGO = (255, 100, 0)
AMARELO = (255, 220, 0)
ROXO_ASTEROIDE = (90, 70, 110)
TEXTO_ASTEROIDE = (255, 235, 180)
COR_BOTAO = (0, 180, 120)
COR_BOTAO_HOVER = (0, 220, 140)

# Fontes arredondadas/modernas
fonte = pygame.font.SysFont("Arial", 36, bold=True)
fonte_pequena = pygame.font.SysFont("Arial", 24, bold=True)

clock = pygame.time.Clock()

# Retângulos dos Botões
botao_iniciar_rect = pygame.Rect(300, 360, 200, 55)
botao_reiniciar_rect = pygame.Rect(300, 430, 200, 55)

# --- CARREGAMENTO DO FUNDO VIA URL ---
IMAGEM_FUNDO = None
try:
    # URL Direta da imagem do Freepik que você escolheu
    url = "https://img.freepik.com/vetores-gratis/fundo-da-galaxia-da-nave-espacial_1308-31420.jpg?w=740"
    
    # Baixa a imagem para a memória do Python de forma segura
    requisicao = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    com_dados = urllib.request.urlopen(requisicao).read()
    imagem_bytes = io.BytesIO(com_dados)
    
    IMAGEM_FUNDO = pygame.image.load(imagem_bytes)
    IMAGEM_FUNDO = pygame.transform.scale(IMAGEM_FUNDO, (LARGURA, ALTURA))
    print("Imagem de fundo carregada com sucesso diretamente da internet!")
except Exception as e:
    print("Não foi possível carregar a imagem da internet, usando o fundo alternativo azul escuro.")

def nova_pergunta():
    a = random.randint(1, 15)
    b = random.randint(1, 15)
    correta = a + b
    respostas = [correta]

    while len(respostas) < 3:
        erro = correta + random.randint(-5, 5)
        if erro > 0 and erro != correta and erro not in respostas:
            respostas.append(erro)

    random.shuffle(respostas)

    asteroides = []
    posicoes_x = [120, 360, 600]
    random.shuffle(posicoes_x)
    
    for x, valor in zip(posicoes_x, respostas):
        asteroides.append({
            "rect": pygame.Rect(x + random.randint(-30, 30), random.randint(-160, -70), 85, 85),
            "valor": valor,
            "cor_detalhe": (random.randint(60, 80), random.randint(45, 60), random.randint(70, 90))
        })

    return a, b, correta, asteroides, time.time()

def reiniciar():
    global nave_x, nave_y, nave_vel_x, nave_vel_y, tiro, pontos, vidas, estado_jogo
    global a, b, correta, asteroides, inicio

    nave_x = 370
    nave_y = 470
    nave_vel_x = 0  
    nave_vel_y = 0  
    tiro = None
    pontos = 0
    vidas = 3
    estado_jogo = "JOGANDO"

    a, b, correta, asteroides, inicio = nova_pergunta()

# Desenha um Foguete 2D Cartoon bem colorido
def desenhar_foguete_2d(x, y, acelerando):
    # Fogo dos motores traseiros (Animado)
    if acelerando and random.choice([True, False]):
        pygame.draw.polygon(tela, LARANJA_FOGO, [(x + 20, y + 65), (x + 40, y + 65), (x + 30, y + 95)])
        pygame.draw.polygon(tela, AMARELO, [(x + 24, y + 65), (x + 36, y + 65), (x + 30, y + 83)])
    elif random.choice([True, False]):
        pygame.draw.polygon(tela, LARANJA_FOGO, [(x + 23, y + 65), (x + 37, y + 65), (x + 30, y + 80)])

    # Asas Laterais (Vermelhas/Laranja)
    pygame.draw.ellipse(tela, LARANJA_FOGO, (x + 2, y + 35, 18, 35))
    pygame.draw.ellipse(tela, LARANJA_FOGO, (x + 40, y + 35, 18, 35))

    # Corpo do Foguete (Cápsula Branca Redondinha)
    pygame.draw.ellipse(tela, BRANCO, (x + 12, y, 36, 70))
    
    # Bico do Foguete (Vermelho)
    pygame.draw.polygon(tela, LARANJA_FOGO, [(x + 13, y + 18), (x + 47, y + 18), (x + 30, y - 5)])

    # Janela Redonda (Azul com brilho)
    pygame.draw.circle(tela, AZUL_PROPULSAO, (x + 30, y + 30), 8)
    pygame.draw.circle(tela, BRANCO, (x + 28, y + 28), 2) # Brilho do vidro

# Desenha um Asteroide Espacial 2D com crateras
def desenhar_asteroide_2d(asteroide):
    rect = asteroide["rect"]
    cx, cy = rect.centerx, rect.centery
    raio = rect.width // 2
    
    # Corpo principal do asteroide
    pygame.draw.circle(tela, ROXO_ASTEROIDE, (cx, cy), raio)
    
    # Desenhar crateras internas para dar textura 2D
    pygame.draw.circle(tela, asteroide["cor_detalhe"], (cx - 15, cy - 15), 10)
    pygame.draw.circle(tela, asteroide["cor_detalhe"], (cx + 18, cy + 10), 12)
    pygame.draw.circle(tela, asteroide["cor_detalhe"], (cx - 5, cy + 20), 8)

estado_jogo = "MENU"
rodando = True

while rodando:
    clock.tick(60)
    
    # Renderiza o Fundo (Imagem da URL ou cor sólida escura)
    if IMAGEM_FUNDO:
        tela.blit(IMAGEM_FUNDO, (0, 0))
    else:
        tela.fill(PRETO)
    
    posicao_mouse = pygame.mouse.get_pos()
    foguete_acelerando = False

    # Eventos
    for evento in pygame.event.get():
        if evento.type == pygame.QUIT:
            rodando = False

        elif evento.type == pygame.MOUSEBUTTONDOWN:
            if evento.button == 1:
                if estado_jogo == "MENU" and botao_iniciar_rect.collidepoint(posicao_mouse):
                    reiniciar()
                elif estado_jogo == "GAME_OVER" and botao_reiniciar_rect.collidepoint(posicao_mouse):
                    reiniciar()

        elif evento.type == pygame.KEYDOWN:
            if estado_jogo == "JOGANDO":
                if evento.key == pygame.K_SPACE and tiro is None:
                    # Laser azul moderno saindo da ponta do foguete
                    tiro = pygame.Rect(nave_x + 28, nave_y - 15, 5, 22)

    # Lógica do Jogo Ativo
    if estado_jogo == "JOGANDO":
        teclas = pygame.key.get_pressed()
        
        if teclas[pygame.K_LEFT]:  nave_vel_x -= 0.7
        if teclas[pygame.K_RIGHT]: nave_vel_x += 0.7
        if teclas[pygame.K_UP]:
            nave_vel_y -= 0.7
            foguete_acelerando = True
        if teclas[pygame.K_DOWN]:  nave_vel_y += 0.7

        # Inércia
        nave_vel_x *= 0.90
        nave_vel_y *= 0.90
        
        nave_x += nave_vel_x
        nave_y += nave_vel_y

        # Limites da tela
        nave_x = max(0, min(LARGURA - 60, nave_x))
        nave_y = max(140, min(ALTURA - 95, nave_y))

        # Movimento do Laser
        if tiro:
            tiro.y -= 16
            if tiro.y < 0:
                tiro = None

        # Movimento e colisão dos Asteroides
        asteroide_fugiu = False
        for asteroide in asteroides:
            velocidade_queda = 1.8 + (pontos * 0.03)
            asteroide["rect"].y += velocidade_queda

            if asteroide["rect"].y > ALTURA:
                asteroide_fugiu = True

            # Colisão do tiro com o asteroide
            if tiro and tiro.colliderect(asteroide["rect"]):
                tempo = time.time() - inicio
                if asteroide["valor"] == correta:
                    if tempo <= 4: pontos += 20
                    elif tempo <= 7: pontos += 10
                    else: pontos += 5
                else:
                    vidas -= 1

                tiro = None
                if vidas <= 0:
                    estado_jogo = "GAME_OVER"
                else:
                    a, b, correta, asteroides, inicio = nova_pergunta()
                break
        
        if asteroide_fugiu and estado_jogo == "JOGANDO":
            vidas -= 1
            if vidas <= 0:
                estado_jogo = "GAME_OVER"
            else:
                a, b, correta, asteroides, inicio = nova_pergunta()

    # --- DESENHOS DA INTERFACE ---
    
    if estado_jogo == "MENU":
        # Painel centralizado elegante
        pygame.draw.rect(tela, (20, 20, 40), (180, 140, 440, 320), border_radius=15)
        pygame.draw.rect(tela, AMARELO, (180, 140, 440, 320), 3, border_radius=15)
        
        txt_titulo = fonte.render("SPACE MATH BLASTER", True, AMARELO)
        txt_sub = fonte_pequena.render("Modo Voo Livre 2D", True, BRANCO)
        tela.blit(txt_titulo, (205, 180))
        tela.blit(txt_sub, (295, 240))
        
        # Botão Iniciar Moderno (Muda de cor com o mouse em cima)
        cor_at_botao = COR_BOTAO_HOVER if botao_iniciar_rect.collidepoint(posicao_mouse) else COR_BOTAO
        pygame.draw.rect(tela, cor_at_botao, botao_iniciar_rect, border_radius=8)
        tela.blit(fonte_pequena.render("JOGAR", True, BRANCO), (360, 373))

    elif estado_jogo == "GAME_OVER":
        pygame.draw.rect(tela, (40, 10, 10), (180, 140, 440, 370), border_radius=15)
        pygame.draw.rect(tela, LARANJA_FOGO, (180, 140, 440, 370), 3, border_radius=15)
        
        tela.blit(fonte.render("FIM DE JOGO", True, LARANJA_FOGO), (290, 190))
        tela.blit(fonte.render(f"PONTOS: {pontos}", True, BRANCO), (300, 270))
        
        # Botão Reiniciar Moderno
        cor_at_botao = COR_BOTAO_HOVER if botao_reiniciar_rect.collidepoint(posicao_mouse) else COR_BOTAO
        pygame.draw.rect(tela, cor_at_botao, botao_reiniciar_rect, border_radius=8)
        tela.blit(fonte_pequena.render("REINICIAR", True, BRANCO), (340, 443))

    elif estado_jogo == "JOGANDO":
        # Painel Superior (HUD) Semitransparente moderno
        hud_surface = pygame.Surface((LARGURA, 125), pygame.SRCALPHA)
        hud_surface.fill((10, 10, 30, 200)) # Azul escuro com transparência
        tela.blit(hud_surface, (0, 0))
        pygame.draw.line(tela, AZUL_PROPULSAO, (0, 125), (LARGURA, 125), 2)
        
        tela.blit(fonte_pequena.render(f"PONTOS: {pontos}", True, AMARELO), (25, 20))
        tela.blit(fonte_pequena.render(f"VIDAS: {vidas}", True, LARANJA_FOGO), (635, 20))
        tela.blit(fonte.render(f"RESOVA: {a} + {b} = ?", True, BRANCO), (230, 65))

        # Desenhar Asteroides e Números
        for asteroide in asteroides:
            desenhar_asteroide_2d(asteroide)
            valor = fonte.render(str(asteroide["valor"]), True, TEXTO_ASTEROIDE)
            # Centralização dinâmica do texto
            v_width = valor.get_width()
            tela.blit(valor, (asteroide["rect"].centerx - v_width // 2, asteroide["rect"].centery - 18))

        # Desenhar Foguete Moderno
        desenhar_foguete_2d(int(nave_x), int(nave_y), foguete_acelerando)

        # Desenhar Laser
        if tiro:
            pygame.draw.rect(tela, AZUL_PROPULSAO, tiro, border_radius=2)

    pygame.display.flip()

pygame.display.quit()
pygame.quit()
print("Processo finalizado.")

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html
Imagem de fundo carregada com sucesso diretamente da internet!
Processo finalizado.
